# Label Analytics — Setup Detector Feedback Loop

Analyze human labels collected via the `/label` UI.
Computes per-type accuracy, false-alert clusters, labeling velocity,
and stop-criteria checks.

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

RULE_VERSION = "v1"
ARTIFACTS_ROOT = Path("../artifacts/setup_review") / RULE_VERSION

manifest_path = ARTIFACTS_ROOT / "manifest.json"
labels_path = ARTIFACTS_ROOT / "labels.json"

with open(manifest_path) as f:
    manifest = json.load(f)

labels_raw = []
if labels_path.exists():
    with open(labels_path) as f:
        labels_raw = json.load(f)

df_manifest = pd.DataFrame(manifest)
df_labels = pd.DataFrame(labels_raw) if labels_raw else pd.DataFrame(columns=["chart_id", "human_label", "correct_type", "reviewed_at"])

if not df_labels.empty:
    df_labels["reviewed_at"] = pd.to_datetime(df_labels["reviewed_at"])

df = df_manifest.merge(df_labels, on="chart_id", how="left")
df["is_labeled"] = df["human_label"].notna()

print(f"Manifest: {len(df_manifest)} charts")
print(f"Labels:   {len(df_labels)} entries")
print(f"Unlabeled: {(~df['is_labeled']).sum()}")

## Per-Type Breakdown

In [ ]:
labeled = df[df["is_labeled"]].copy()

if labeled.empty:
    print("No labels yet. Start labeling at /label in the frontend.")
else:
    summary = labeled.groupby("setup_type").agg(
        total=("chart_id", "count"),
        yes_count=("human_label", lambda x: (x == "yes").sum()),
        no_count=("human_label", lambda x: (x == "no").sum()),
        wrong_type_count=("human_label", lambda x: (x == "wrong_type").sum()),
        unsure_count=("human_label", lambda x: (x == "unsure").sum()),
    ).reset_index()

    summary["yes_rate"] = (summary["yes_count"] / summary["total"] * 100).round(1)
    summary["no_rate"] = (summary["no_count"] / summary["total"] * 100).round(1)
    summary["wrong_type_rate"] = (summary["wrong_type_count"] / summary["total"] * 100).round(1)

    print(summary.to_string(index=False))
    print(f"\nOverall yes-rate: {labeled['human_label'].eq('yes').mean()*100:.1f}%")
    print(f"Overall no-rate:  {labeled['human_label'].eq('no').mean()*100:.1f}%")

## Yes/No Rate by Setup Type — Bar Chart

In [ ]:
if not labeled.empty:
    fig = go.Figure()
    for label_val, color, name in [
        ("yes", "#22c55e", "Yes"),
        ("no", "#ef4444", "No"),
        ("wrong_type", "#f59e0b", "Wrong Type"),
        ("unsure", "#6b7280", "Unsure"),
    ]:
        counts = labeled[labeled["human_label"] == label_val].groupby("setup_type").size()
        fig.add_trace(go.Bar(
            x=counts.index,
            y=counts.values,
            name=name,
            marker_color=color,
        ))
    fig.update_layout(
        title="Label Distribution by Setup Type",
        barmode="stack",
        template="plotly_dark",
        height=400,
        xaxis_title="Setup Type",
        yaxis_title="Count",
    )
    fig.show()

## False-Alert Clusters

Group `No` labels by ticker + window to find systematic failures.

In [ ]:
if not labeled.empty:
    false_alerts = labeled[labeled["human_label"] == "no"].copy()
    if false_alerts.empty:
        print("No false alerts found yet.")
    else:
        clusters = false_alerts.groupby(["ticker", "setup_type", "window_start", "window_end"]).agg(
            false_count=("chart_id", "count"),
        ).reset_index().sort_values("false_count", ascending=False)

        print("Top false-alert clusters (ticker + type + window):")
        print(clusters.head(20).to_string(index=False))

        by_type = false_alerts.groupby("setup_type").size().sort_values(ascending=False)
        print(f"\nFalse alerts by type:\n{by_type.to_string()}")

        by_ticker = false_alerts.groupby("ticker").size().sort_values(ascending=False)
        print(f"\nFalse alerts by ticker (top 10):\n{by_ticker.head(10).to_string()}")

## Wrong-Type Analysis

What types are being confused with each other?

In [ ]:
if not labeled.empty:
    wrong = labeled[labeled["human_label"] == "wrong_type"].copy()
    if wrong.empty:
        print("No wrong-type labels yet.")
    else:
        confusion = wrong.groupby(["setup_type", "correct_type"]).size().reset_index(name="count")
        confusion = confusion.sort_values("count", ascending=False)
        print("Type confusion matrix (predicted -> correct):")
        print(confusion.to_string(index=False))

## Labeling Velocity

In [ ]:
if not df_labels.empty and "reviewed_at" in df_labels.columns:
    df_labels["review_date"] = df_labels["reviewed_at"].dt.date
    daily = df_labels.groupby("review_date").size().reset_index(name="labels")
    daily["cumulative"] = daily["labels"].cumsum()

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(
        x=daily["review_date"].astype(str),
        y=daily["labels"],
        name="Daily",
        marker_color="#3b82f6",
    ), secondary_y=False)
    fig.add_trace(go.Scatter(
        x=daily["review_date"].astype(str),
        y=daily["cumulative"],
        name="Cumulative",
        line=dict(color="#f59e0b", width=2),
    ), secondary_y=True)
    fig.update_layout(
        title="Labeling Velocity",
        template="plotly_dark",
        height=350,
    )
    fig.update_yaxes(title_text="Labels/Day", secondary_y=False)
    fig.update_yaxes(title_text="Cumulative", secondary_y=True)
    fig.show()

    total_labeled = len(df_labels)
    days_active = max(1, (df_labels["reviewed_at"].max() - df_labels["reviewed_at"].min()).days + 1)
    rate = total_labeled / days_active
    remaining = len(df_manifest) - total_labeled
    est_days = remaining / rate if rate > 0 else float("inf")
    print(f"Average: {rate:.0f} labels/day")
    print(f"Remaining: {remaining} charts")
    print(f"Estimated completion: {est_days:.0f} days at current pace")
else:
    print("No labels with timestamps yet.")

## Holdout Window Validation

Split windows into tuning vs holdout sets. The most recent half-year window
per ticker is held out. Compare yes-rates between tuning and holdout to
detect overfitting.

In [ ]:
if not labeled.empty and "window_end" in labeled.columns:
    labeled["window_end_dt"] = pd.to_datetime(labeled["window_end"])

    holdout_mask = pd.Series(False, index=labeled.index)
    for ticker in labeled["ticker"].unique():
        ticker_mask = labeled["ticker"] == ticker
        max_window = labeled.loc[ticker_mask, "window_end_dt"].max()
        holdout_mask |= (ticker_mask & (labeled["window_end_dt"] == max_window))

    tuning = labeled[~holdout_mask]
    holdout = labeled[holdout_mask]

    print(f"Tuning set:  {len(tuning)} labels")
    print(f"Holdout set: {len(holdout)} labels")
    print()

    if not tuning.empty and not holdout.empty:
        for stype in sorted(labeled["setup_type"].unique()):
            t_sub = tuning[tuning["setup_type"] == stype]
            h_sub = holdout[holdout["setup_type"] == stype]
            t_yes = (t_sub["human_label"] == "yes").mean() * 100 if len(t_sub) > 0 else 0
            h_yes = (h_sub["human_label"] == "yes").mean() * 100 if len(h_sub) > 0 else 0
            delta = h_yes - t_yes
            flag = " *** OVERFIT RISK ***" if delta < -10 else ""
            print(f"{stype:40s}  tuning={t_yes:5.1f}%  holdout={h_yes:5.1f}%  delta={delta:+5.1f}%{flag}")

        t_overall = (tuning["human_label"] == "yes").mean() * 100
        h_overall = (holdout["human_label"] == "yes").mean() * 100
        print(f"\n{'OVERALL':40s}  tuning={t_overall:5.1f}%  holdout={h_overall:5.1f}%  delta={h_overall-t_overall:+5.1f}%")
    else:
        print("Need labels in both tuning and holdout windows to compare.")
else:
    print("No labeled data with window information available.")

## Rule Delta Proposal Summary

Generate a structured summary of false-positive patterns and type confusion
that can be used by Cursor Agent to propose specific rule parameter changes.

In [ ]:
if not labeled.empty:
    rule_cfg_path = Path("rule_configs") / f"{RULE_VERSION}.json"
    current_rules = {}
    if rule_cfg_path.exists():
        with open(rule_cfg_path) as f:
            current_rules = json.load(f).get("parameters", {})

    proposal = {
        "current_version": RULE_VERSION,
        "current_parameters": current_rules,
        "total_labeled": len(labeled),
        "overall_yes_rate": round((labeled["human_label"] == "yes").mean() * 100, 1),
        "overall_no_rate": round((labeled["human_label"] == "no").mean() * 100, 1),
        "per_type_stats": {},
        "top_false_alert_clusters": [],
        "type_confusion": [],
    }

    for stype in labeled["setup_type"].unique():
        sub = labeled[labeled["setup_type"] == stype]
        proposal["per_type_stats"][stype] = {
            "total": len(sub),
            "yes_rate": round((sub["human_label"] == "yes").mean() * 100, 1),
            "no_rate": round((sub["human_label"] == "no").mean() * 100, 1),
        }

    false_alerts = labeled[labeled["human_label"] == "no"]
    if not false_alerts.empty:
        clusters = false_alerts.groupby(["ticker", "setup_type"]).size().reset_index(name="count")
        clusters = clusters.sort_values("count", ascending=False).head(10)
        proposal["top_false_alert_clusters"] = clusters.to_dict("records")

    wrong = labeled[labeled["human_label"] == "wrong_type"]
    if not wrong.empty:
        confusion = wrong.groupby(["setup_type", "correct_type"]).size().reset_index(name="count")
        proposal["type_confusion"] = confusion.to_dict("records")

    proposal_path = ARTIFACTS_ROOT / "rule_delta_proposal.json"
    with open(proposal_path, "w") as f:
        json.dump(proposal, f, indent=2, default=str)

    print(f"Rule delta proposal saved to {proposal_path}")
    print(json.dumps(proposal, indent=2, default=str)[:2000])
else:
    print("No labels available for rule delta proposal.")

## Stop Criteria Check

Per setup type: if yes-rate is stable within +/-2% for 3 consecutive
batches of 100 labels, that type is **saturated** and further labeling
provides diminishing returns.

In [ ]:
BATCH_SIZE = 100
STABILITY_THRESHOLD = 0.02
REQUIRED_STABLE_BATCHES = 3

if not labeled.empty:
    saturated_types = []
    for stype in labeled["setup_type"].unique():
        subset = labeled[labeled["setup_type"] == stype].sort_values("reviewed_at").reset_index(drop=True)
        n = len(subset)
        if n < BATCH_SIZE * REQUIRED_STABLE_BATCHES:
            print(f"{stype}: {n} labels — need {BATCH_SIZE * REQUIRED_STABLE_BATCHES} for saturation check")
            continue

        batch_rates = []
        for i in range(0, n, BATCH_SIZE):
            batch = subset.iloc[i : i + BATCH_SIZE]
            if len(batch) < BATCH_SIZE:
                break
            rate = (batch["human_label"] == "yes").mean()
            batch_rates.append(rate)

        if len(batch_rates) >= REQUIRED_STABLE_BATCHES:
            recent = batch_rates[-REQUIRED_STABLE_BATCHES:]
            spread = max(recent) - min(recent)
            is_saturated = spread <= STABILITY_THRESHOLD
            status = "SATURATED" if is_saturated else "NOT SATURATED"
            if is_saturated:
                saturated_types.append(stype)
            print(f"{stype}: {status} (last {REQUIRED_STABLE_BATCHES} batches spread={spread:.3f}, rates={[f'{r:.2f}' for r in recent]})")
        else:
            print(f"{stype}: {len(batch_rates)} full batches — need {REQUIRED_STABLE_BATCHES}")

    all_types = labeled["setup_type"].unique()
    if len(saturated_types) == len(all_types) and len(all_types) > 0:
        print("\n*** ALL TYPES SATURATED — ready for Phase 3 rule tuning ***")
    elif saturated_types:
        print(f"\nSaturated: {saturated_types}")
        print(f"Still need labels: {[t for t in all_types if t not in saturated_types]}")
    else:
        print("\nNo types saturated yet. Keep labeling!")
else:
    print("No labels yet.")

## Rule Version Comparison

Compare yes-rates across rule versions (run after creating v2+ configs).

In [ ]:
import glob

versions_root = Path("../artifacts/setup_review")
version_dirs = sorted([d.name for d in versions_root.iterdir() if d.is_dir() and (d / "manifest.json").exists()]) if versions_root.exists() else []

if len(version_dirs) > 1:
    comparison = []
    for v in version_dirs:
        m_path = versions_root / v / "manifest.json"
        l_path = versions_root / v / "labels.json"
        with open(m_path) as f:
            m = json.load(f)
        labels_v = []
        if l_path.exists():
            with open(l_path) as f:
                labels_v = json.load(f)
        total = len(m)
        labeled_v = len(labels_v)
        yes_v = sum(1 for l in labels_v if l.get("human_label") == "yes")
        comparison.append({
            "version": v,
            "total_charts": total,
            "labeled": labeled_v,
            "yes_count": yes_v,
            "yes_rate": f"{yes_v/labeled_v*100:.1f}%" if labeled_v > 0 else "n/a",
        })
    print(pd.DataFrame(comparison).to_string(index=False))
else:
    print(f"Only {len(version_dirs)} version(s) found. Create v2+ rule configs and re-run batch export to compare.")